# Single-attribute FDs that hold in hospital_2k

For every ordered pair of distinct columns `A -> B`, check whether the FD holds
exactly in `clean.csv` (using `g3_error == 0`, the standard G3 measure from §3.1).

An FD also holds trivially whenever the LHS column has no duplicate values: each
LHS group then has exactly one row, so *any* `A -> B` is satisfied. Those are not
real dependencies, they are an artefact of `A` being a (near-)key. The combination
check below keeps an FD only when it **holds AND its LHS has duplicates**.

**Inputs → Outputs:** `datasets/hospital_2k/{clean.csv, fds.csv}` → a `results`
table of every candidate single-attr FD (holds / lhs_has_dups / meaningful / g3),
plus a diff of the discovered "meaningful" set against the declared `fds.csv`.

**Config knobs:**
- `DATASET` — dataset folder under `datasets/`.
- `EPS` — tolerance below which G3 counts as an exact hold.

In [ ]:
import sys
from pathlib import Path

# horizon/ pipeline modules use flat imports (`from fds.fd import ...`) and need
# horizon/ on sys.path; eval + package imports (`horizon.fds.fd`) need the repo
# root. Put both on the path, repo root first.
REPO = Path("..").resolve()
HZN = REPO / "horizon"
for p in (str(HZN), str(REPO)):
    if p in sys.path:
        sys.path.remove(p)
sys.path.insert(0, str(HZN))
sys.path.insert(0, str(REPO))

import polars as pl

from horizon.fds.fd import FunctionalDependency
from horizon.utils.loaders import get_fds, load_table
from eval.fd_eval import g3_error

In [ ]:
# --- config: dataset folder under datasets/ ---
DATASET = Path("..") / "datasets" / "hospital_2k"
csv_path = DATASET / "clean.csv"
fds_path = DATASET / "fds.csv"

df = load_table(csv_path)
print(f"{df.height} rows, {df.width} cols")
df.columns

In [ ]:
# Check every ordered pair A -> B of distinct single columns.
#   holds       : the FD is satisfied exactly (G3 == 0)
#   lhs_has_dups : the LHS column has at least one repeated value
#   meaningful   : holds AND lhs_has_dups  (not a key-artefact FD)
# g3_error trusts a pre-projected df (see its docstring), so pass df[[a, b]].
EPS = 1e-9

rows = []
for a in df.columns:
    for b in df.columns:
        if a == b:
            continue
        fd = FunctionalDependency(a, b)
        sub = df.select([a, b]).drop_nulls()
        n = sub.height
        lhs_n_unique = sub.select(a).n_unique()
        g3 = g3_error(sub, fd)
        holds = g3 < EPS
        lhs_has_dups = lhs_n_unique < n
        rows.append(
            {
                "lhs": a,
                "rhs": b,
                "holds": holds,
                "lhs_has_dups": lhs_has_dups,
                "meaningful": holds and lhs_has_dups,
                "g3_error": g3,
                "lhs_n_unique": lhs_n_unique,
                "n": n,
            }
        )

results = pl.DataFrame(rows)
print(
    f"{results.height} candidate FDs | "
    f"{results['holds'].sum()} hold | "
    f"{results['meaningful'].sum()} hold with LHS duplicates (meaningful)"
)

## All FDs that hold exactly (`g3 == 0`)

`lhs_has_dups = false` flags the trivial ones that only hold because the LHS is a (near-)key.

In [ ]:
with pl.Config(tbl_rows=-1):
    print(results.filter(pl.col("holds")).sort(["lhs", "rhs"]))

## Combination check: holds AND LHS has duplicates

These are the single-attribute FDs that are genuinely supported by repeated LHS values.

In [ ]:
with pl.Config(tbl_rows=-1):
    print(results.filter(pl.col("meaningful")).sort(["lhs", "rhs"]))

## How the discovered set compares to the declared `fds.csv`

Single-attribute FDs only (the declared set also has composite-LHS FDs, which this scan does not enumerate).

In [ ]:
declared = get_fds(fds_path, csv_path)
declared_single = {(fd.lhs, fd.rhs) for fd in declared if len(fd.lhs_attributes) == 1}
discovered = {
    (r["lhs"], r["rhs"]) for r in results.filter(pl.col("meaningful")).iter_rows(named=True)
}

print("declared single-attr FDs:", sorted(declared_single))
print()
print("declared & discovered:", sorted(declared_single & discovered))
print("declared, NOT discovered:", sorted(declared_single - discovered))
print("discovered, NOT declared:", sorted(discovered - declared_single))